# **Super resolution x4 model training**

In [1]:
import torch
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
# !pip install lpips -q
import lpips

# LPIPS network loaded once at module level (expensive to initialize)
_lpips_net = None

def _get_lpips():
    global _lpips_net
    if _lpips_net is None:
        _lpips_net = lpips.LPIPS(net="vgg")
    return _lpips_net


def sne(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Squared Norm Error — lower is better."""
    return torch.sum((pred - target) ** 2).item()


def psnr(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0) -> float:
    """Peak Signal-to-Noise Ratio — higher is better."""
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float("inf")
    return 10 * torch.log10(torch.tensor(max_val ** 2 / mse)).item()


def ssim(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Structural Similarity Index — higher is better."""
    return ssim_fn(pred, target, data_range=1.0).item()


def lpips_score(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Learned Perceptual Image Patch Similarity — lower is better."""
    net = _get_lpips()
    net.eval()
    # lpips expects images in [-1, 1]
    pred_n   = pred   * 2 - 1
    target_n = target * 2 - 1
    with torch.no_grad():
        return net(pred_n, target_n).mean().item()


def compute_all(pred: torch.Tensor, target: torch.Tensor) -> dict:
    """Compute all four metrics at once. Tensors shape: (B, C, H, W), range [0, 1]."""
    return {
        "SNE":   sne(pred, target),
        "PSNR":  psnr(pred, target),
        "SSIM":  ssim(pred, target),
        "LPIPS": lpips_score(pred, target),
    }


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.5 MB/s eta 0:00:00


### Model architecture

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.down = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1)
        self.act  = nn.LeakyReLU(0.2, inplace=True)
        self.res  = ResBlock(out_ch)

    def forward(self, x):
        return self.res(self.act(self.down(x)))


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up  = nn.Conv2d(in_ch, out_ch * 4, kernel_size=3, padding=1)
        self.ps  = nn.PixelShuffle(2)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(self.ps(self.up(x)))


class SRUNet(nn.Module):
    def __init__(self, base_ch: int = 32, n_bottleneck: int = 4):
        super().__init__()
        C = base_ch

        self.head = nn.Sequential(
            nn.Conv2d(3, C, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(C),
        )

        self.enc1 = DownBlock(C, C * 2)
        self.enc2 = DownBlock(C * 2, C * 4)

        self.bottleneck = nn.Sequential(
            *[ResBlock(C * 4) for _ in range(n_bottleneck)]
        )

        self.up1  = UpBlock(C * 4, C * 2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(C * 4, C * 2, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(C * 2),
        )

        self.up2  = UpBlock(C * 2, C)
        self.dec2 = nn.Sequential(
            nn.Conv2d(C * 2, C, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(C),
        )

        self.sr_up1 = UpBlock(C, C)
        self.sr_up2 = UpBlock(C, C)
        self.tail   = nn.Conv2d(C, 3, kernel_size=3, padding=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skip_global = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False)

        s0 = self.head(x)
        s1 = self.enc1(s0)
        e  = self.enc2(s1)

        b = self.bottleneck(e)

        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, s0], dim=1))

        out = self.sr_up1(d2)
        out = self.sr_up2(out)
        out = self.tail(out)

        return out + skip_global

In [3]:
model = SRUNet(base_ch=32, n_bottleneck=4)
x = torch.zeros(1, 3, 64, 64)
print(model(x).shape)  # torch.Size([1, 3, 256, 256])
print(sum(p.numel() for p in model.parameters()), "params")

torch.Size([1, 3, 256, 256])
2289923 params


### Train

In [20]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from PIL import Image
import random
import time
import numpy as np

# from metrics import psnr

# config
EPOCHS         = 3
VAL_EVERY      = 1
SAVE_EVERY     = 10
BATCH_SIZE     = 32
LIMIT          = None  # set to None to use all data
LR             = 1e-4
LR_STEP        = 30
LR_GAMMA       = 0.5
BASE_CH        = 32
N_BOTTLENECK   = 4
NUM_WORKERS    = 4
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_LR_DIR   = Path("/kaggle/input/datasets/gpla77/prepared-data-train/lr_x4/lr_x4")
TRAIN_HR_DIR   = Path("/kaggle/input/datasets/gpla77/prepared-data-train/hr/hr")
VALID_LR_DIR   = Path("/kaggle/input/datasets/gpla77/prepared-data-valid/lr_x4/lr_x4")
VALID_HR_DIR   = Path("/kaggle/input/datasets/gpla77/prepared-data-valid/hr/hr")

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# dataset
class SRDataset(Dataset):
    def __init__(self, lr_dir: Path, hr_dir: Path, limit: int = None, shuffle: bool = False):
        pairs = list(zip(sorted(lr_dir.glob("*.png")), sorted(hr_dir.glob("*.png"))))
        if shuffle:
            random.shuffle(pairs)
        if limit is not None:
            pairs = pairs[:limit]
        self.lr_paths, self.hr_paths = zip(*pairs)

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr = TF.to_tensor(Image.open(self.lr_paths[idx]).convert("RGB"))
        hr = TF.to_tensor(Image.open(self.hr_paths[idx]).convert("RGB"))
        return lr, hr



# validation
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    with torch.no_grad():
        for lr_batch, hr_batch in loader:
            lr_batch = lr_batch.to(device)
            hr_batch = hr_batch.to(device)
            pred = model(lr_batch).clamp(0, 1)
            total_loss += criterion(pred, hr_batch).item()
            for i in range(pred.size(0)):
                p = psnr(pred[i].unsqueeze(0), hr_batch[i].unsqueeze(0))
                if p != float("inf"):
                    total_psnr += p
                    n += 1
    return total_loss / len(loader), total_psnr / max(n, 1)

In [21]:
from tqdm.notebook import tqdm

print(f"Device: {DEVICE}")

train_ds = SRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, limit=LIMIT, shuffle=True)
valid_ds  = SRDataset(VALID_LR_DIR, VALID_HR_DIR, limit=LIMIT, shuffle=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)}")

model     = SRUNet(base_ch=BASE_CH, n_bottleneck=N_BOTTLENECK).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)
criterion = nn.L1Loss()

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

best_psnr = 0.0

for epoch in tqdm(range(1, EPOCHS + 1), desc="Epochs"):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for lr_batch, hr_batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        lr_batch = lr_batch.to(DEVICE)
        hr_batch = hr_batch.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(lr_batch), hr_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    print(f"Epoch [{epoch:>3}/{EPOCHS}]  loss: {epoch_loss/len(train_loader):.5f}"
            f"  lr: {scheduler.get_last_lr()[0]:.2e}  time: {time.time()-t0:.1f}s")

    if epoch % VAL_EVERY == 0:
        val_loss, val_psnr = validate(model, valid_loader, criterion, DEVICE)
        marker = " ← best" if val_psnr > best_psnr else ""
        print(f"  ┗ VAL  loss: {val_loss:.5f}  PSNR: {val_psnr:.4f} dB{marker}")
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            torch.save(model.state_dict(), CHECKPOINT_DIR / "best.pth")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            "epoch":     epoch,
            "model":     model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_psnr": best_psnr,
        }, CHECKPOINT_DIR / f"epoch_{epoch:04d}.pth")

tqdm.write(f"\nBest PSNR: {best_psnr:.4f} dB")

Device: cuda
Train: 32000 | Valid: 3598
Params: 2,289,923


Epochs:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch [  1/3]  loss: 0.09452  lr: 1.00e-04  time: 91.3s
  ┗ VAL  loss: 0.03090  PSNR: 28.7346 dB ← best


Epoch 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch [  2/3]  loss: 0.03142  lr: 1.00e-04  time: 88.7s
  ┗ VAL  loss: 0.02993  PSNR: 29.2396 dB ← best


Epoch 3:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch [  3/3]  loss: 0.03103  lr: 1.00e-04  time: 88.9s
  ┗ VAL  loss: 0.02980  PSNR: 29.3317 dB ← best

Best PSNR: 29.3317 dB
